In [ ]:
%%sql -r CreateSemanticView
USE ROLE LOGISTICS_HOL_ADMIN;
USE DATABASE LOGISTICS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE HOL_WH;

CREATE OR REPLACE SEMANTIC VIEW LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV

  TABLES (
    shipments AS LOGISTICS_HOL.ANALYTICS.SHIPMENTS
      PRIMARY KEY (SHIPMENT_ID)
      COMMENT = 'One row per shipment with carrier, route, delivery performance, cost.'
  )

  FACTS (
    shipments.late_flag AS IFF(is_late, 1, 0)
      COMMENT = 'One when the shipment was late, zero otherwise.',
    shipments.on_time_flag AS IFF(is_late, 0, 1)
      COMMENT = 'One when the shipment was on time, zero otherwise.',
    shipments.delay AS delay_days
      COMMENT = 'Number of days the shipment was delayed.',
    shipments.cost AS cost_usd
      COMMENT = 'Shipping cost in US dollars.',
    shipments.weight AS weight_kg
      COMMENT = 'Package weight in kilograms.'
  )

  DIMENSIONS (
    shipments.shipment_id AS shipment_id
      COMMENT = 'Unique shipment identifier.',
    shipments.carrier AS carrier
      WITH SYNONYMS ('shipping company', 'delivery company', 'courier')
      COMMENT = 'Carrier: FedEx, UPS, DHL, or USPS.',
    shipments.service_type AS service_type
      WITH SYNONYMS ('shipping speed', 'service level')
      COMMENT = 'Express, Standard Ground, or Overnight.',
    shipments.origin_city AS origin_city COMMENT = 'City the shipment originated from.',
    shipments.origin_state AS origin_state COMMENT = 'State the shipment originated from.',
    shipments.destination_city AS destination_city COMMENT = 'Destination city.',
    shipments.destination_state AS destination_state COMMENT = 'Destination state.',
    shipments.status AS status COMMENT = 'Delivered or Delayed.',
    shipments.customer_segment AS customer_segment
      WITH SYNONYMS ('segment', 'customer type')
      COMMENT = 'Enterprise, Retail, Healthcare, or SMB.',
    shipments.region AS region COMMENT = 'US region of the delivery.',
    shipments.quarter AS quarter COMMENT = 'Fiscal quarter of the scheduled delivery.',
    shipments.scheduled_delivery AS scheduled_delivery COMMENT = 'Scheduled delivery date.'
  )

  METRICS (
    shipments.total_shipments AS COUNT(shipments.shipment_id)
      COMMENT = 'Total number of shipments.',
    shipments.late_shipments AS SUM(shipments.late_flag)
      COMMENT = 'Number of late shipments.',
    shipments.on_time_shipments AS SUM(shipments.on_time_flag)
      COMMENT = 'Number of on-time shipments.',
    shipments.on_time_rate AS AVG(shipments.on_time_flag)
      COMMENT = 'Fraction of shipments delivered on time (0 to 1).',
    shipments.avg_delay_days AS AVG(shipments.delay)
      COMMENT = 'Average delay in days across shipments.',
    shipments.total_cost AS SUM(shipments.cost)
      COMMENT = 'Total shipping cost in US dollars.'
  )

  COMMENT = 'Semantic view for logistics shipment analytics, used by Cortex Analyst.';

SELECT 'Semantic view LOGISTICS_ANALYTICS_SV created' AS STATUS;
     

In [ ]:
%%sql -r DescribeSV
DESCRIBE SEMANTIC VIEW LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV;

In [ ]:
%%sql -r OnTimeByCarrier
-- The headline metric: on-time rate by carrier
SELECT * FROM SEMANTIC_VIEW(
  LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV
  DIMENSIONS shipments.carrier
  METRICS shipments.total_shipments, shipments.on_time_rate, shipments.avg_delay_days
)
ORDER BY on_time_rate ASC;

In [ ]:
%%sql -r DhlByRegion
-- Drill into DHL by region
SELECT * FROM SEMANTIC_VIEW(
  LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV
  DIMENSIONS shipments.carrier, shipments.region
  METRICS shipments.total_shipments, shipments.late_shipments, shipments.on_time_rate
)
WHERE carrier = 'DHL'
ORDER BY on_time_rate ASC;
     

In [ ]:
%%sql -r SegmentView
-- On-time rate by customer segment (which customers feel the pain most)
SELECT * FROM SEMANTIC_VIEW(
  LOGISTICS_HOL.AGENTS.LOGISTICS_ANALYTICS_SV
  DIMENSIONS shipments.carrier, shipments.customer_segment
  METRICS shipments.on_time_rate
)
WHERE carrier = 'DHL'
ORDER BY on_time_rate ASC;